# Putting it together: your grade + the material

Now we connect the two systems. The portal knows your grade on a quiz. The CMS
holds the quiz and its model answer. One agent can use both: find the quiz you
did worst on, open its solution, and walk you through the topic.

Be honest about the limit. The agent can see your grade and the course material,
but not your own answer sheet. So it cannot say "you wrote X". What it can do is
find your weak spot and re-teach it. That is still a real study helper.

## 1. Log in (both systems, one login)

This works for **GIU** as well: put `GUC_SITE=giu` in your `.env` and use your
GIU username and password. Nothing else in the notebook changes.

In [ ]:
import os, getpass
from dotenv import load_dotenv

load_dotenv()  # reads ANTHROPIC_API_KEY from .env

# One GUC login works for both the portal and the CMS. getpass hides the
# password so it is never saved in the notebook.
if not os.environ.get("GUC_USERNAME"):
    os.environ["GUC_USERNAME"] = input("GUC username: ")
if not os.environ.get("GUC_PASSWORD"):
    os.environ["GUC_PASSWORD"] = getpass.getpass("GUC password: ")

from guc_portal import GucPortal
from guc_cms import GucCms

portal = GucPortal()
cms = GucCms()
print("logged in to both the portal and the CMS")

## 2. Tools from both packages

Three tools, two of them from the CMS package and one from the portal:

- `get_my_grades` (portal): find the weak quiz
- `list_course_files` (CMS): find the matching quiz or solution file
- `read_course_file` (CMS): pull its content into the chat

In [ ]:
from io import BytesIO
from markitdown import MarkItDown
from langchain.tools import tool

converter = MarkItDown()
_grades_cache = {}


@tool
def get_my_grades(semester: str, course: str) -> dict:
    """Get the student's marks for one course in one term (from the portal).

    `semester` like 'Winter 2024', `course` like 'Discrete Math'. The portal is
    slow, so call this once.
    """
    key = (semester.lower(), course.lower())
    if key not in _grades_cache:
        g = portal.get_grades_by_name(semester, course)
        _grades_cache[key] = {
            "course": g.course,
            "term": g.season,
            "items": [{"what": i.assessment, "grade": i.grade} for i in g.items],
        }
    return _grades_cache[key]


@tool
def list_course_files(course: str) -> list[dict]:
    """List files in a CMS course (title + kind), to find a quiz or its solution.

    `course` is a name or code like 'Discrete Math' or 'MATH501'.
    """
    c = cms.find_course(course)
    if not c:
        return [{"error": f"no course matches {course!r}"}]
    return [{"title": i.title, "kind": i.kind} for i in cms.get_content(c).items]


@tool
def read_course_file(course: str, file_title: str) -> str:
    """Read a CMS file's content as text, so you can discuss what is inside it.

    `file_title` can be part of the title, e.g. 'Quiz 2 Model Answers'.
    """
    c = cms.find_course(course)
    if not c:
        return f"no course matches {course!r}"
    item = next(
        (i for i in cms.get_content(c).items if file_title.lower() in i.title.lower()),
        None,
    )
    if not item:
        return f"no file matching {file_title!r}"
    raw = cms.fetch_bytes(item)
    ext = "." + item.filename.rsplit(".", 1)[-1].lower()
    return converter.convert_stream(BytesIO(raw), file_extension=ext).text_content[:18000]

## 3. The study-helper agent

In [ ]:
from langchain.agents import create_agent
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage

model = ChatAnthropic(model="claude-haiku-4-5", temperature=0)

agent = create_agent(
    model=model,
    tools=[get_my_grades, list_course_files, read_course_file],
    system_prompt=(
        "You are a study helper for a GUC student. When they ask about a quiz or "
        "assignment they struggled with: "
        "1) use get_my_grades to find their marks and the weakest item; "
        "2) use list_course_files to find that quiz's solution or model answer; "
        "3) use read_course_file to read it; "
        "4) explain what the quiz covered and where students usually lose marks. "
        "You can see the grade and the material, not the student's own answers, so "
        "focus on re-teaching the topic. The portal is slow: call get_my_grades once."
    ),
)

question = (
    "I think I did badly on a quiz in Discrete Math in Winter 2024. "
    "Which quiz was it, and can you pull its model answers and explain what it tested?"
)
print(agent.invoke({"messages": [HumanMessage(question)]})["messages"][-1].content)

## What just happened, and the point

One agent, two systems. It read your grade from the portal, saw which quiz was
weakest, found that quiz's model answers in the CMS, read them, and explained the
topic. You never told it which quiz or which file. It found them.

This is the payoff of small, honest tools: each does one plain job, and the agent
combines them into something neither system does on its own.

Your turn:

- Ask it to compare two quizzes.
- Point it at a different course or term.